In [2]:
from pathlib import Path
import pandas as pd

# Define the project root directory
PROJECT_ROOT = Path.cwd().parent

# Locate the raw-data directory
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Find all historical CSV files
csv_files = sorted(DATA_DIR.glob("barentswatch_lice_*.csv"))

# Display the results
print(f"Total CSV files found: {len(csv_files)}")

for file in csv_files:
    print(file.name)

Total CSV files found: 11
barentswatch_lice_2015.csv
barentswatch_lice_2016.csv
barentswatch_lice_2017.csv
barentswatch_lice_2018.csv
barentswatch_lice_2019.csv
barentswatch_lice_2020.csv
barentswatch_lice_2021.csv
barentswatch_lice_2022.csv
barentswatch_lice_2023.csv
barentswatch_lice_2024.csv
barentswatch_lice_2025.csv


## Check whether all 11 CSV files (2015–2025) contain the same column names in the same order.

In [3]:
# Use the first CSV file as our reference
reference_columns = pd.read_csv(csv_files[0], nrows=0).columns.tolist()

# Compare every CSV file against the reference
for file in csv_files:
    current_columns = pd.read_csv(file, nrows=0).columns.tolist()

    if current_columns == reference_columns:
        print(f"✅ {file.name}: MATCH")
    else:
        print(f"❌ {file.name}: SCHEMA MISMATCH")
        print("Expected:", reference_columns)
        print("Found:", current_columns)

✅ barentswatch_lice_2015.csv: MATCH
✅ barentswatch_lice_2016.csv: MATCH
✅ barentswatch_lice_2017.csv: MATCH
✅ barentswatch_lice_2018.csv: MATCH
✅ barentswatch_lice_2019.csv: MATCH
✅ barentswatch_lice_2020.csv: MATCH
✅ barentswatch_lice_2021.csv: MATCH
✅ barentswatch_lice_2022.csv: MATCH
✅ barentswatch_lice_2023.csv: MATCH
✅ barentswatch_lice_2024.csv: MATCH
✅ barentswatch_lice_2025.csv: MATCH


In [4]:
# Load the 2015 dataset
df_2015 = pd.read_csv(csv_files[0], low_memory=False)

# Count missing values in each column
missing_values = df_2015.isna().sum()

# Calculate the percentage of missing values
missing_percentage = (
    df_2015.isna().mean() * 100
)

# Create a summary table
missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percentage
})

# Display columns with missing values
missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
]

display(missing_summary.round(2))

,Missing Count,Missing Percentage
Lokalitetsnavn,13,0.02
Voksne hunnlus,29780,52.15
Lus i bevegelige stadier,29780,52.15
Fastsittende lus,29780,52.15
Kommunenummer,13,0.02
Kommune,13,0.02
Fylkesnummer,13,0.02
Fylke,13,0.02
Lat,13,0.02
Lon,13,0.02


## Investigate why lice measurements are missing
**Objective:** 
* Determine whether missing sea-lice measurements are associated with localities reporting no fish or no lice counting.

In [5]:
# Identify records with missing adult female lice measurements
missing_lice = df_2015["Voksne hunnlus"].isna()

# Examine reporting status for those records
missing_lice_summary = pd.crosstab(
    df_2015.loc[missing_lice, "Trolig uten fisk"],
    df_2015.loc[missing_lice, "Har telt lakselus"],
    margins=True
)

display(missing_lice_summary)

Har telt lakselus,Nei,All
Trolig uten fisk,,
Ja,27988,27988
Nei,1792,1792
All,29780,29780


In [6]:
# Check duplicate observations
# Define the columns expected to identify a weekly observation
key_columns = ["Lokalitetsnummer", "År", "Uke"]

# Find records sharing the same locality-year-week combination
duplicate_mask = df_2015.duplicated(
    subset=key_columns,
    keep=False
)

# Select the potentially duplicated records
duplicate_records = df_2015.loc[duplicate_mask]

# Display summary
print("Total observations:", len(df_2015))
print("Rows with duplicate keys:", len(duplicate_records))
print("Completely identical rows:", df_2015.duplicated().sum())

# Show examples if duplicates exist
display(duplicate_records.head(10))

Total observations: 57105
Rows with duplicate keys: 0
Completely identical rows: 0


,Uke,År,Lokalitetsnummer,Lokalitetsnavn,Voksne hunnlus,Lus i bevegelige stadier,Fastsittende lus,Trolig uten fisk,Har telt lakselus,Kommunenummer,Kommune,Fylkesnummer,Fylke,Lat,Lon,Lusegrense uke,Over lusegrense uke,Sjøtemperatur,ProduksjonsområdeId,Produksjonsområde


In [7]:
# Check duplicates across all 11 years
# Columns that should uniquely identify each observation
key_columns = ["Lokalitetsnummer", "År", "Uke"]

# Store validation results
duplicate_results = []

# Check each historical dataset
for file in csv_files:

    df = pd.read_csv(file, low_memory=False)

    # Identify duplicated locality-year-week combinations
    duplicate_count = df.duplicated(
        subset=key_columns,
        keep=False
    ).sum()

    # Count completely identical records
    identical_count = df.duplicated().sum()

    # Store the results
    duplicate_results.append({
        "Year": int(df["År"].iloc[0]),
        "Total Rows": len(df),
        "Duplicate Keys": duplicate_count,
        "Identical Rows": identical_count
    })

# Convert results into a DataFrame
duplicate_summary = pd.DataFrame(duplicate_results)

# Display the summary
display(duplicate_summary)

,Year,Total Rows,Duplicate Keys,Identical Rows
0,2015,57105,0,0
1,2016,56041,0,0
2,2017,56424,0,0
3,2018,54925,0,0
4,2019,54947,0,0
5,2020,56354,0,0
6,2021,55491,0,0
7,2022,55454,0,0
8,2023,55606,0,0
9,2024,55608,0,0


## Validate Primary Key Columns
* **Objective:**  Check whether Lokalitetsnummer, År, and Uke contain missing values across all 11 datasets.

In [8]:
# Define the proposed primary key columns
key_columns = ["Lokalitetsnummer", "År", "Uke"]

# Create an empty list for validation results
key_validation = []

# Check each dataset
for file in csv_files:

    df = pd.read_csv(file, low_memory=False)

    # Count missing values in each key column
    missing_counts = df[key_columns].isna().sum()

    # Store the results
    key_validation.append({
        "Year": int(df["År"].iloc[0]),
        "Missing Locality ID": missing_counts["Lokalitetsnummer"],
        "Missing Year": missing_counts["År"],
        "Missing Week": missing_counts["Uke"]
    })

# Convert results into a DataFrame
key_summary = pd.DataFrame(key_validation)

display(key_summary)

,Year,Missing Locality ID,Missing Year,Missing Week
0,2015,0,0,0
1,2016,0,0,0
2,2017,0,0,0
3,2018,0,0,0
4,2019,0,0,0
5,2020,0,0,0
6,2021,0,0,0
7,2022,0,0,0
8,2023,0,0,0
9,2024,0,0,0


In [9]:
## Merged all data sets.
# Read all 11 CSV files into a list of DataFrames
dataframes = [
    pd.read_csv(file, low_memory=False)
    for file in csv_files
]

# Combine the DataFrames vertically
df_combined = pd.concat(
    dataframes,
    axis=0,
    ignore_index=True
)

In [10]:
# Display the combined dataset information
print("Total observations:", len(df_combined))
print("Total columns:", len(df_combined.columns))
print("Years:", sorted(df_combined["År"].unique()))

# Display the first five records
display(df_combined.head())

Total observations: 613601
Total columns: 20
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,Uke,År,Lokalitetsnummer,Lokalitetsnavn,Voksne hunnlus,Lus i bevegelige stadier,Fastsittende lus,Trolig uten fisk,Har telt lakselus,Kommunenummer,Kommune,Fylkesnummer,Fylke,Lat,Lon,Lusegrense uke,Over lusegrense uke,Sjøtemperatur,ProduksjonsområdeId,Produksjonsområde
0,53,2015,15196,Aga Ø,0.00,0.03,0.06,Nei,Ja,1219.0,BØMLO,12.0,Hordaland,59.845100,5.261367,0.5,Nei,7.5,3.0,Karmøy til Sotra
1,53,2015,12067,Aldalen,0.10,0.43,0.10,Nei,Ja,1243.0,OS I HORDALAND,12.0,Hordaland,60.249950,5.571917,0.5,Nei,1.0,3.0,Karmøy til Sotra
2,53,2015,12982,Aldeøyna,0.20,1.25,0.00,Nei,Ja,1428.0,ASKVOLL,14.0,Sogn og Fjordane,61.310250,4.769050,0.5,Nei,7.8,4.0,Nordhordaland til Stadt
3,53,2015,11756,Allersholmen,0.26,0.42,0.10,Nei,Ja,1264.0,AUSTRHEIM,12.0,Hordaland,60.774650,4.853600,0.5,Nei,7.6,4.0,Nordhordaland til Stadt
4,53,2015,13271,Almurden,NaN,NaN,NaN,Ja,Nei,1749.0,FLATANGER,17.0,Nord-Trøndelag,64.433235,10.680950,0.5,NaN,NaN,7.0,Nord-Trøndelag med Bindal


In [11]:
# Check data types before Saving
# Examine the data types before saving
dtype_summary = pd.DataFrame({
    "Column": df_combined.columns,
    "Data Type": df_combined.dtypes.astype(str).values,
    "Missing Values": df_combined.isna().sum().values
})

display(dtype_summary)
    

,Column,Data Type,Missing Values
0,Uke,int64,0
1,År,int64,0
2,Lokalitetsnummer,int64,0
3,Lokalitetsnavn,str,127
4,Voksne hunnlus,float64,296166
5,Lus i bevegelige stadier,float64,296166
6,Fastsittende lus,float64,296166
7,Trolig uten fisk,str,0
8,Har telt lakselus,str,0
9,Kommunenummer,float64,127


In [12]:
#Examine the unique values in the regulatory thresholds column
threshold_values = df_combined["Lusegrense uke"].value_counts(dropna=False)
display(threshold_values)

Lusegrense uke
0.5       457486
0.5        98368
0.2        44774
0.2        12846
Ukjent       127
Name: count, dtype: int64

In [13]:
# Identify the Actual Value Types
# Objective: Determine whether the column contains both numeric values and strings.
# Select the regulatory threshold column
threshold_column = df_combined["Lusegrense uke"]

# Identify the Python type of each value
value_types = threshold_column.map(type)

# Count each data type
type_summary = value_types.value_counts()

# Display the results
display(type_summary)

Lusegrense uke
<class 'str'>      502387
<class 'float'>    111214
Name: count, dtype: int64

In [14]:
# Convert the Regulatory Threshold to Numeric
# Objective: Convert Lusegrense uke to float64, treating Ukjent as a missing value.
# Preserve the original regulatory threshold values
df_combined["Lusegrense_original"] = df_combined["Lusegrense uke"]

# Convert the threshold column to numeric
df_combined["Lusegrense uke"] = pd.to_numeric(
    df_combined["Lusegrense uke"],
    errors="coerce"
)

# Verify the conversion
print("Data type:", df_combined["Lusegrense uke"].dtype)

print("\nThreshold values:")
print(df_combined["Lusegrense uke"].value_counts(dropna=False))

print("\nMissing values:")
print(df_combined["Lusegrense uke"].isna().sum())

Data type: float64

Threshold values:
Lusegrense uke
0.5    555854
0.2     57620
NaN       127
Name: count, dtype: int64

Missing values:
127


In [15]:
# Verify the Conversion
# Before saving our combined dataset as Parquet, we should confirm that no valid numeric thresholds were accidentally lost during conversion.
# Identify rows where conversion produced a missing value
missing_threshold = df_combined["Lusegrense uke"].isna()

# Examine the original values for those rows
original_missing_values = df_combined.loc[
    missing_threshold,
    "Lusegrense_original"
]

# Count original values that could not be converted
print("Original values converted to NaN:")
print(original_missing_values.value_counts(dropna=False))

# Verify the expected unknown-value count
assert missing_threshold.sum() == 127, (
    "Unexpected number of missing thresholds!"
)

print("\nValidation passed!")

Original values converted to NaN:
Lusegrense_original
Ukjent    127
Name: count, dtype: int64

Validation passed!


In [17]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 49.0 MB/s  0:00:01 eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Convert original regulatory thresholds to a consistent string type
df_combined["Lusegrense_original"] = (
    df_combined["Lusegrense_original"].astype("string")
)

# Verify the new data type
print("Data type:", df_combined["Lusegrense_original"].dtype)

# Display unique values and their frequencies
print("\nOriginal threshold values:")
print(df_combined["Lusegrense_original"].value_counts(dropna=False))

Data type: string

Original threshold values:
Lusegrense_original
0.5       555854
0.2        57620
Ukjent       127
Name: count, dtype: int64[pyarrow]


In [18]:
# Save the Combined Dataset as Parquet
# Objective: Save all 613,601 observations from 2015–2025 in a single Parquet file.
# Define the output directory
output_dir = PROJECT_ROOT / "data" / "processed"

# Create the directory if it does not exist
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output filename
output_file = output_dir / "barentswatch_lice_2015_2025.parquet"

# Save the combined dataset
df_combined.to_parquet(
    output_file,
    engine="pyarrow",
    index=False
)

# Confirm that the file was saved
print("File saved:", output_file.name)
print("File size:", round(output_file.stat().st_size / 1024**2, 2), "MB")

File saved: barentswatch_lice_2015_2025.parquet
File size: 5.78 MB


In [19]:
# Validate the paraquet file
# Read the saved Parquet file
df_check = pd.read_parquet(output_file)

# Verify the number of rows
assert len(df_check) == len(df_combined), "Row count mismatch!"

# Verify the number of columns
assert len(df_check.columns) == len(df_combined.columns), "Column count mismatch!"

# Verify the column names and order
assert df_check.columns.tolist() == df_combined.columns.tolist(), (
    "Column names do not match!"
)

# Display the results
print("Parquet validation successful!")
print("Total observations:", len(df_check))
print("Total columns:", len(df_check.columns))
print("Years:", sorted(df_check["År"].unique()))

Parquet validation successful!
Total observations: 613601
Total columns: 21
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
